Created Train/Test split with the latest month (September 2026) being the test split and the year before being the train split.

In [98]:
from pathlib import Path
import os
import pandas as pd

DATA_DIR = Path(os.getenv("IDX_DATA_DIR", "data"))
CSV_DIR = DATA_DIR / "raw/csv"

from pathlib import Path
import os
import pandas as pd

DATA_DIR = Path(os.getenv("IDX_DATA_DIR", "data"))
CSV_DIR = DATA_DIR / "raw/csv"

TEST_MONTH = "202609"
TRAIN_MONTHS = [
    "202509", "202510", "202511", "202512",
    "202601", "202602", "202603", "202604",
    "202605", "202606", "202607", "202608",
]

In [99]:
def sold_path(month):
    generated = CSV_DIR / f"GeneratedCRMLSSold{month}.csv"
    raw = CSV_DIR / f"CRMLSSold{month}.csv"
    if generated.exists():
        return generated
    return raw


def load_month(month):
    df = pd.read_csv(sold_path(month), low_memory=False)
    df["month"] = month
    return df


train = pd.concat([load_month(m) for m in TRAIN_MONTHS], ignore_index=True)
test = load_month(TEST_MONTH)

print("train months", TRAIN_MONTHS)
print(train["month"].value_counts().sort_index())
print(test["month"].value_counts().sort_index())

train months ['202509', '202510', '202511', '202512', '202601', '202602', '202603', '202604', '202605', '202606', '202607', '202608']
month
202509    22443
202510    23233
202511    19088
202512    20538
202601    16487
202602    19010
202603    23372
202604    24261
202605    24277
202606    25812
202607    25467
202608    23133
Name: count, dtype: int64
month
202609    22107
Name: count, dtype: int64


Combining test/train tables into one in order to handle null values and encode categorical columns. 

In [100]:
df = pd.concat(
    [train.assign(split="train"), test.assign(split="test")],
    ignore_index=True,
)
df["split"].value_counts()

split
train    267121
test      22107
Name: count, dtype: int64

Keep the home features used for modeling, columns relating to listing itself are not included

In [101]:
keep_cols = [
    "ClosePrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "GarageSpaces",
    "PropertyType",
    "PropertySubType",
    "WaterfrontYN",
    "BasementYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "PoolPrivateYN",
    "ViewYN",
    "NewConstructionYN",
    "City",
    "PostalCode",
    "Latitude",
    "Longitude",
    "month",
    "Flooring",
    "FireplacesTotal",
    "AssociationFeeFrequency",
    "CountyOrParish",
    "ElementarySchool",
    "ParkingTotal",
    "PropertySubType",
    "LotSizeAcres",
    "BuildingAreaTotal",
    "ElementarySchoolDistrict",
    "StateOrProvince",
    "BusinessType",
    "CoveredSpaces",
    "MiddleOrJuniorSchool",
    "Stories",
    "HighSchool",
    "Levels",
    "HighSchoolDistrict",
    "AssociationFee",
    "MiddleOrJuniorSchoolDistrict",
    "split",
] 
df = df[keep_cols]

Determining the proportion of entries that have a null value for each column

In [102]:
null_pct = df.isnull().mean() * 100
null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
null_pct

MiddleOrJuniorSchoolDistrict    100.000000
FireplacesTotal                 100.000000
CoveredSpaces                   100.000000
ElementarySchoolDistrict        100.000000
WaterfrontYN                     99.943989
BusinessType                     99.742072
BasementYN                       98.363229
ElementarySchool                 89.583650
MiddleOrJuniorSchool             89.481655
BuildingAreaTotal                86.603303
HighSchool                       86.197740
AssociationFeeFrequency          70.411578
Flooring                         41.415423
HighSchoolDistrict               34.165433
AssociationFee                   30.470079
AttachedGarageYN                 28.433969
Stories                          17.907671
GarageSpaces                     12.800974
NewConstructionYN                12.000221
Levels                           11.499924
PoolPrivateYN                    10.729252
ViewYN                           10.268370
LotSizeAcres                      8.551385
FireplaceYN

Drop columns that are null for every row, observe remaining columns

In [103]:
all_null = df.columns[df.isna().all()]
df = df.drop(columns=all_null)

# Dropping null Close Price rows because they are not useful for modeling,
# Dropping null State/Province rows because of low null percentages
df = df.dropna(subset=["ClosePrice", "StateOrProvince"]) 

null_pct = df.isnull().mean() * 100
null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
print(null_pct.to_string())


WaterfrontYN               99.943987
BusinessType               99.742066
BasementYN                 98.363189
ElementarySchool           89.584090
MiddleOrJuniorSchool       89.482092
BuildingAreaTotal          86.602978
HighSchool                 86.198098
AssociationFeeFrequency    70.411208
Flooring                   41.415734
HighSchoolDistrict         34.165223
AssociationFee             30.470125
AttachedGarageYN           28.433274
Stories                    17.907413
GarageSpaces               12.800938
NewConstructionYN          11.999820
Levels                     11.499511
PoolPrivateYN              10.729166
ViewYN                     10.268272
LotSizeAcres                8.551592
FireplaceYN                 8.336186
LotSizeSquareFeet           8.297115
PropertySubType             7.141943
PropertySubType             7.141943
LivingArea                  6.816241
BedroomsTotal               6.355002
BathroomsTotalInteger       4.354110
YearBuilt                   4.005242
P

See the values a column takes, including null

For waterfront column, notice that the only non-null value is true meaning we can impute the null values to be false and then encode the columns by making true = 1 and false = 0

In [104]:
def colValues(col):
    counts = df[col].value_counts(dropna=False)
    print(f"{len(counts)} distinct values")
    print(counts.to_string())

colValues("WaterfrontYN")

2 distinct values
WaterfrontYN
NaN     289059
True       162


In [105]:
df["WaterfrontYN"] = df["WaterfrontYN"].eq(True).astype(int)
df["BasementYN"] = df["BasementYN"].eq(True).astype(int)

For LotSize, avg sqft col and acres (in sqft) col, if one is null take the other measurement. If both are null flag in the missing column and set to 0. 

In [106]:
sqft = df["LotSizeSquareFeet"]
acres_sqft = df["LotSizeAcres"] * 43560
lot = (sqft.fillna(acres_sqft) + acres_sqft.fillna(sqft)) / 2

df["LotSizeSquareFeet_missing"] = lot.isna().astype(int)
df["LotSizeSquareFeet"] = lot.fillna(0)
df = df.drop(columns=["LotSizeAcres"])

For other YN columns, 1 for True, 0 for False, and 0 for null and flag in a col_missing column where 0 is not missing and 1 is missing

Flag other columns missing that have high null percentages

In [107]:
yn_cols = [
    "AttachedGarageYN",
    "FireplaceYN",
    "PoolPrivateYN",
    "ViewYN",
    "NewConstructionYN",
]

for col in yn_cols:
    df[f"{col}_missing"] = df[col].isna().astype(int)
    df[col] = df[col].map({True: 1, False: 0}).fillna(0).astype(int)

flag_cols = [
    "GarageSpaces",
    "Levels",
    "Stories",
    "AssociationFee",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "YearBuilt",
    "ParkingTotal",
]

for col in flag_cols:
    df[f"{col}_missing"] = df[col].isna().astype(int)
    df[col] = df[col].fillna(0)

df.head(2)

,ClosePrice,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,YearBuilt,GarageSpaces,PropertyType,PropertySubType,WaterfrontYN,...,NewConstructionYN_missing,GarageSpaces_missing,Levels_missing,Stories_missing,AssociationFee_missing,LivingArea_missing,BedroomsTotal_missing,BathroomsTotalInteger_missing,YearBuilt_missing,ParkingTotal_missing
0,35000.0,0.0,0.0,0.0,108900.0,0.0,0.0,Land,NaN,0,...,0,1,1,1,0,1,1,1,1,1
1,90000.0,0.0,0.0,0.0,653400.0,0.0,0.0,Land,NaN,0,...,0,1,1,1,1,1,1,1,1,1
